In [ ]:
import sys
import os
import io
import contextlib

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_all_runs, load_top_n_runs,
    plot_2d_histograms, plot_hp_sensitivity,
    print_hp_sensitivity_table, print_robustness_table,
    get_optimizer_colors,
    TASK_CONFIGS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

BACKEND = "local"
RESULTS_DIR = os.path.join('..', '..', 'results')

# Optional: filter to a subset of optimizers (None = all)
OPTIMIZERS_TO_PLOT = None  # e.g. ['adam', 'sgd_learn_diag_curv']


def _quiet(fn, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)


def show_hp_sensitivity(task_key):
    """Load and show all HP sensitivity analysis for a sweep task."""
    cfg = TASK_CONFIGS[task_key]
    optimizers = OPTIMIZERS_TO_PLOT or cfg['optimizers']
    colors = get_optimizer_colors(optimizers)
    name = cfg['display_name']
    itr = cfg['iteration']

    all_runs = _quiet(load_all_runs,
        backend=BACKEND, optimizers=optimizers,
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        iteration=itr,
    )
    top_n_runs = _quiet(load_top_n_runs,
        backend=BACKEND, optimizers=optimizers, n=50,
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        metric_key=cfg['metric_key'], direction=cfg['direction'],
        sort_metric=cfg['sort_metric'], sort_order=cfg['sort_order'],
        iteration=itr,
    )
    print(f"Loaded {sum(len(v) for v in all_runs.values())} total runs")

    fig = plot_2d_histograms(
        top_n_runs, backend=BACKEND,
        x_metric=cfg['hist_x'], y_metric=cfg['hist_y'],
        optimizers=optimizers,
    )
    plt.show()

    for opt in optimizers:
        runs = all_runs.get(opt, [])
        if not runs:
            continue
        kwargs = dict(metric_key=cfg['metric_key'], direction=cfg['direction'])
        threshold = cfg.get('convergence_threshold')
        if threshold is not None:
            kwargs['convergence_threshold'] = threshold
        fig, axes = plot_hp_sensitivity(runs, opt, **kwargs)
        plt.show()

    for opt in optimizers:
        runs = all_runs.get(opt, [])
        if len(runs) >= 10:
            print_hp_sensitivity_table(runs, metric_key=cfg['metric_key'],
                                       title=f'HP Sensitivity — {opt}')

    print_robustness_table(all_runs, metric_key=cfg['metric_key'],
                           direction=cfg['direction'],
                           title=f'{name} (itr {itr}) — Robustness')

# HP Sensitivity Analysis

Sweep distributions, HP sensitivity scatter grids, Spearman correlation tables, and robustness metrics.
Set `OPTIMIZERS_TO_PLOT` above to filter to a subset.

## MNIST MLP

In [ ]:
show_hp_sensitivity("mnist_mlp")

## CIFAR-10 ResNet-18

In [ ]:
show_hp_sensitivity("cifar10_resnet18")

## Shakespeare MiniGPT

In [ ]:
show_hp_sensitivity("shakespeare_minigpt")

## Regression

In [ ]:
show_hp_sensitivity("regression")

## Small Examples

In [ ]:
FUNCTIONS = ['beale', 'rosenbrock', 'himmelblau', 'ackley', 'rastrigin', 'styblinski_tang']
ITERATION_SE = 4

if BACKEND == "local":
    _avail = set()
    for fn in FUNCTIONS:
        td = os.path.join(RESULTS_DIR, f"small_examples_{fn}")
        if os.path.isdir(td):
            _avail.update(d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d)))
    SE_OPTIMIZERS = sorted(_avail)
else:
    SE_OPTIMIZERS = ['adam', 'sgd_learn_diag', 'sgd_learn_diag_curv']

optimizers = OPTIMIZERS_TO_PLOT or SE_OPTIMIZERS

# Pool runs across functions for HP sensitivity
pooled_all = {}
pooled_top = {}
for fn in FUNCTIONS:
    tag = f"small_examples_{fn}"
    ar = _quiet(load_all_runs, backend=BACKEND, optimizers=optimizers,
               task_tag=tag, results_dir=RESULTS_DIR, iteration=ITERATION_SE)
    tn = _quiet(load_top_n_runs, backend=BACKEND, optimizers=optimizers, n=50,
               task_tag=tag, results_dir=RESULTS_DIR,
               metric_key="sweep_metric", direction="minimize",
               sort_metric="sweep_metric", sort_order="+", iteration=ITERATION_SE)
    for opt in optimizers:
        pooled_all.setdefault(opt, []).extend(ar.get(opt, []))
        pooled_top.setdefault(opt, []).extend(tn.get(opt, []))

print(f"Loaded {sum(len(v) for v in pooled_all.values())} total runs (pooled across {len(FUNCTIONS)} functions)")

for opt in optimizers:
    runs = pooled_all.get(opt, [])
    if not runs:
        continue
    fig, axes = plot_hp_sensitivity(runs, opt,
                                    metric_key="sweep_metric", direction="minimize")
    plt.show()

for opt in optimizers:
    runs = pooled_all.get(opt, [])
    if len(runs) >= 10:
        print_hp_sensitivity_table(runs, metric_key="sweep_metric",
                                   title=f'HP Sensitivity — {opt} (pooled small examples)')

print_robustness_table(pooled_all, metric_key="sweep_metric", direction="minimize",
                       title=f'Small Examples (pooled, itr {ITERATION_SE}) — Robustness')